# 05 章 / 第 1 节 · SFT 数据准备:Alpaca-zh + chat template + loss mask

## 学习目标
- 把 Alpaca-zh(instruction / input / output)正确转成 **Qwen2.5 chat template** 格式
- 理解 **损失 mask** 的必要性(prompt 部分不算 loss)
- 学会用 `DataCollatorForCompletionOnlyLM` 自动 mask,或手写 label 函数
- 验证 mask 正确性(打印 `labels` 看 prompt 是不是全 -100)

## 对应八股
`llm-interview/llm-ft.md`(数据章)


## 1. 概念背景

### 1.1 SFT 的最小要素

一条 SFT 样本 = `(prompt, response)`,目标:让模型在 prompt 后**生成 response**。
训练时:
- input_ids = `[prompt_tokens + response_tokens]`
- labels    = `[-100 × len(prompt) + response_tokens]`  ← 关键!

`-100` 是 PyTorch `CrossEntropyLoss` 的 ignore_index,不算 loss。

### 1.2 不 mask prompt 会怎样?

模型会同时学着"复读问题",严重浪费样本。
经验:同样数据,带 mask 收敛快 2-3x,效果好 5-10pt。

### 1.3 Qwen2.5 chat template 长什么样

```
<|im_start|>system\n{system}<|im_end|>
<|im_start|>user\n{user}<|im_end|>
<|im_start|>assistant\n{response}<|im_end|>
```

**永远用 `tokenizer.apply_chat_template`,不要手写**,版本间格式可能微调。


## 2. 环境检查


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from scripts.env_check import check
check()


## 3. 加载 tokenizer + 看 chat template


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"tokenizer 类型: {type(tokenizer).__name__}")
print(f"vocab size: {tokenizer.vocab_size}")
print(f"chat template(前 200 字符):\n{tokenizer.chat_template[:200]}...")


## 4. 加载 Alpaca-zh 并应用 chat template


In [ ]:
from datasets import load_dataset

raw = load_dataset("shibing624/alpaca-zh", split="train")
print(f"原始样本: {len(raw)}")
print(f"\n--- 字段示例 ---")
print(raw[0])


In [ ]:
def to_messages(example):
    """Alpaca 三段 → ChatML messages 列表。"""
    instr = example["instruction"]
    inp   = example.get("input", "") or ""
    user_msg = instr if not inp else f"{instr}\n\n{inp}"
    return {
        "messages": [
            {"role": "user",      "content": user_msg},
            {"role": "assistant", "content": example["output"]},
        ]
    }

ds = raw.shuffle(seed=42).select(range(500)).map(to_messages, remove_columns=raw.column_names)
print(f"格式化后样本: {len(ds)}")
print(f"\n--- 一条 messages ---")
for m in ds[0]["messages"]:
    print(f"[{m['role']}] {m['content'][:100]}")


## 5. 演示 1:用 `apply_chat_template` + 手动 mask


In [ ]:
def format_with_mask(example):
    msgs = example["messages"]
    # 整段:user + assistant
    full_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    # 仅 prompt 部分:user + assistant header(无 response)
    prompt_text = tokenizer.apply_chat_template(msgs[:1], tokenize=False, add_generation_prompt=True)

    full_ids   = tokenizer(full_text,   add_special_tokens=False)["input_ids"]
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]

    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    # 截断到 prompt 长度可能不完全匹配(罕见),按 full 截
    labels = labels[:len(full_ids)]
    return {"input_ids": full_ids, "labels": labels}


sample = format_with_mask(ds[0])
print(f"input_ids 长度: {len(sample['input_ids'])}")
print(f"labels 长度:   {len(sample['labels'])}")
print(f"\nlabels 前 30 个(应大部分是 -100): {sample['labels'][:30]}")
print(f"labels 后 30 个(应是真实 token id): {sample['labels'][-30:]}")
print(f"\nmask 比例(prompt/total): {sum(1 for l in sample['labels'] if l == -100) / len(sample['labels']) * 100:.1f}%")


## 6. 演示 2:用 `DataCollatorForCompletionOnlyLM`(TRL 推荐)


In [ ]:
from trl import DataCollatorForCompletionOnlyLM

# Qwen2.5 chat template 里 response 紧跟在 "<|im_start|>assistant\n" 之后
response_template = "<|im_start|>assistant\n"

# 让 dataset 给我们 text 字符串
def to_text(example):
    full = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    return {"text": full}

ds_text = ds.map(to_text, remove_columns=ds.column_names)

collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)

# tokenize 几条来看
encoded = [tokenizer(x["text"], return_tensors="pt", add_special_tokens=False) for x in ds_text.select(range(2))]
features = [{"input_ids": e["input_ids"][0]} for e in encoded]
batch = collator(features)

print(f"batch input_ids shape: {batch['input_ids'].shape}")
print(f"batch labels shape:    {batch['labels'].shape}")
print(f"第 0 条 labels 前 30:  {batch['labels'][0, :30].tolist()}")
print(f"第 0 条 labels 后 30:  {batch['labels'][0, -30:].tolist()}")


## 7. 踩坑记录

- **手写 mask 时 `add_special_tokens=False`**:Qwen tokenizer 默认会加 BOS,手算 prompt 长度会偏移
- **`apply_chat_template(..., add_generation_prompt=True)` 用于推理时**:它会在结尾加 `<|im_start|>assistant\n` 但不补 `<|im_end|>`,等模型生成
- **`response_template` 必须和 chat template 字面一致**:Qwen 是 `<|im_start|>assistant\n`(带换行),Llama-3 是 `<|start_header_id|>assistant<|end_header_id|>\n\n`,搞错 collator 找不到分隔点,整段全 mask
- **多轮对话**:`messages = [user, assistant, user, assistant]` 时只 mask 第一个 prompt 不够,要每个 assistant 段都不 mask、user 段都 mask;TRL 0.12+ 的 `DataCollatorForCompletionOnlyLM` 已正确处理
- **打 batch 前看一眼 labels**:这是 SFT 最常见的 silent bug,一旦上线训练几小时才发现 OOM 之外没改善,亏死

## 8. 延伸阅读

- [TRL DataCollator 文档](https://huggingface.co/docs/trl/sft_trainer#train-on-completions-only)
- [Qwen2.5 chat template 源码](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/blob/main/tokenizer_config.json)
- [[Slipbox/chat-template-and-loss-mask]]
- 本仓:[`03_unsloth_lora.ipynb`](03_unsloth_lora.ipynb)(主推 SFT 路径)
